In [2]:
#importar librerias
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

#Cargar las zonas
ruta_zonas = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/Zonas_Medellin.gpkg"
zonas = gpd.read_file(ruta_zonas)

print(f" Zonas cargadas: {len(zonas)}")
print(f"   Urbanas: {len(zonas[zonas['Tipo'] == 'Urbano'])}")
print(f"   Rurales: {len(zonas[zonas['Tipo'] == 'Rural'])}")

# Cargar imagenes LST
ruta_lst = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/LST_Reproyectadas/"
archivos_lst = [f for f in os.listdir(ruta_lst) if f.endswith('.tif')]
archivos_lst.sort()

print(f"\n Imágenes LST encontradas: {len(archivos_lst)}")
print("Primeras 5 imágenes:")
for archivo in archivos_lst[:5]:
    print(f"   - {archivo}")

# Extraer datos de temperatura por zona para cada año
resultados = []

for archivo in archivos_lst:
    # Extraer el año del nombre del archivo
    try:
        año = int(archivo.split('_')[1])
    except:
        año = 0
        print(f" No se pudo extraer el año de: {archivo}")
    
    ruta_raster = os.path.join(ruta_lst, archivo)
    print(f"Procesando: {archivo} (Año: {año})")
    
    # Extraer estadísticas zonales
    stats = zonal_stats(
        zonas.geometry,           # Polígonos
        ruta_raster,              # Imagen LST
        stats=['mean', 'max', 'min', 'count'],
        nodata=-9999              # Valor que representa "sin datos"
    )
    
    # Guardar resultados en una lista
    for i, stat in enumerate(stats):
        # Asegurarse de que no sea None
        if stat is not None and stat['mean'] is not None:
            resultados.append({
                'ID': i,
                'Nombre': zonas.iloc[i].get('Nombre', f'Zona_{i}'),
                'Tipo': zonas.iloc[i]['Tipo'],
                'Año': año,
                'Temp_media': stat['mean'],
                'Temp_max': stat['max'],
                'Temp_min': stat['min'],
                'Num_pixeles': stat['count']
            })

# Generar dataframe y guardar
df_temp = pd.DataFrame(resultados)

print(f"\n Temperatura extraída: {len(df_temp)} registros")
print("\nPrimeras 5 filas:")
print(df_temp.head())

# Guardar el archivo CSV
ruta_csv = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/Temperatura_por_zona.csv"
df_temp.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
print(f"\n Archivo guardado en: {ruta_csv}")

# Estadisticos iniciales
print("\n Estadísticas de temperatura por tipo de zona:")
print(df_temp.groupby('Tipo')['Temp_media'].describe())

 Zonas cargadas: 332
   Urbanas: 271
   Rurales: 61

 Imágenes LST encontradas: 23
Primeras 5 imágenes:
   - LST_2013_07_16_9377.tif
   - LST_2014_07_19_9377.tif
   - LST_2014_08_20_9377.tif
   - LST_2015_01_11_9377.tif
   - LST_2015_12_29_9377.tif
Procesando: LST_2013_07_16_9377.tif (Año: 2013)
Procesando: LST_2014_07_19_9377.tif (Año: 2014)
Procesando: LST_2014_08_20_9377.tif (Año: 2014)
Procesando: LST_2015_01_11_9377.tif (Año: 2015)
Procesando: LST_2015_12_29_9377.tif (Año: 2015)
Procesando: LST_2016_05_21_9377.tif (Año: 2016)
Procesando: LST_2016_06_22_9377.tif (Año: 2016)
Procesando: LST_2017_05_24_9377.tif (Año: 2017)
Procesando: LST_2017_12_18_9377.tif (Año: 2017)
Procesando: LST_2018_04_09_9377.tif (Año: 2018)
Procesando: LST_2019_07_17_9377.tif (Año: 2019)
Procesando: LST_2020_01_09_9377.tif (Año: 2020)
Procesando: LST_2020_02_10_9377.tif (Año: 2020)
Procesando: LST_2021_12_13_9377.tif (Año: 2021)
Procesando: LST_2022_01_30_9377.tif (Año: 2022)
Procesando: LST_2022_07_09_9377